# Foundation Models
## TimeGPT, Chronos y el futuro del forecasting

> **Fase 3 — Video 17** | Modelado y Pronóstico
> Dataset: Histórico de Ventas Sintético

---

### Modelos que pronostican sin entrenar

Los **foundation models** para series de tiempo son el equivalente a los LLM: redes (transformers)
pre-entrenadas en **millones de series** que pronostican **zero-shot** — sin ajustar nada a tus datos. Le
pasas tu historia y te devuelven el futuro, en segundos, sin feature engineering ni selección de orden.

- **TimeGPT** (Nixtla) — el pionero, vía API (de pago).
- **Chronos** (Amazon) — open-source, corre local. Lo usamos aquí, **de verdad**.
- **TimesFM** (Google), **Moirai** (Salesforce) — otros abiertos.

La pregunta honesta que cierra la Fase 3: tras todo lo que construimos, **¿un modelo que no vio nunca tus
datos le gana a tu SARIMAX afinado?**

### Ruta del notebook
1. Librerías y carga
2. ¿Qué es un foundation model?
3. Chronos zero-shot en 3 líneas (con incertidumbre)
4. Zero-shot a escala: cualquier SKU, sin entrenar
5. Veredicto honesto vs. todo lo anterior
6. Cuándo usarlos + nota sobre TimeGPT / N-BEATS
7. Conclusiones y puente a la Fase 4

---
## 1. Librerías y carga

> **Nota.** Chronos usa PyTorch y descarga los pesos del modelo desde Hugging Face la primera vez
> (necesita internet una vez). Usamos `chronos-bolt-small`: ligero y corre en CPU en milisegundos.

In [ ]:
import warnings, logging, sys
warnings.filterwarnings('ignore')
logging.getLogger('transformers').setLevel(logging.ERROR)

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from chronos import BaseChronosPipeline
from pathlib import Path

plt.rcParams.update({
    'figure.facecolor': '#F8FAFC', 'axes.facecolor': '#F8FAFC',
    'axes.grid': True, 'grid.color': '#E2E8F0',
    'grid.linewidth': 0.8, 'font.size': 11,
})
BLUE, RED, GREEN, PURPLE, ORANGE = '#2563EB','#DC2626','#16A34A','#7C3AED','#EA580C'
unit_fmt = FuncFormatter(lambda x, _: f'{x/1e3:.0f}k')
print('✅ Librerías cargadas')

In [ ]:
def find_csv(filename='sales_history.csv', max_levels=4):
    base = Path().resolve()
    for _ in range(max_levels):
        candidate = base / 'output' / filename
        if candidate.exists():
            return candidate
        base = base.parent
    raise FileNotFoundError(f"No se encontró '{filename}'. Corre main.py primero.")

csv_path = find_csv()
sys.path.insert(0, str(csv_path.parents[1]))
from src.metrics import wape, mase

df = pd.read_csv(csv_path, parse_dates=['date']).sort_values('date')
y = df.groupby(pd.Grouper(key='date', freq='W-MON'))['units_sold'].sum().iloc[1:-1]

H, m = 52, 52
train, test = y.iloc[:-H], y.iloc[-H:]
snaive = pd.Series(train.iloc[-m:].values[:H], index=test.index)
print(f'✅ Serie semanal: {len(y)} semanas | holdout {H}')

---
## 2. ¿Qué es un foundation model?

| | Modelos clásicos / ML (V11–V16) | Foundation models |
|---|---|---|
| Entrenamiento | Ajustas a *tu* serie | Pre-entrenado en millones de series |
| Tu dato | Imprescindible para entrenar | Solo como **contexto** (zero-shot) |
| Feature engineering | Manual (Fase 2) | Ninguno |
| Tiempo a un pronóstico | Minutos–horas | **Segundos** |
| Exógenas (promo, precio) | Sí (SARIMAX/XGBoost) | Limitado / no en la versión base |

La promesa: **cero configuración**. Le das la serie y listo. Veámoslo funcionar.

---
## 3. Chronos zero-shot en 3 líneas

Cargar el modelo, pasarle el contexto y pedir el pronóstico con **intervalos de incertidumbre** (Chronos
es probabilístico por diseño). **No entrenamos nada.**

In [ ]:
pipe = BaseChronosPipeline.from_pretrained('amazon/chronos-bolt-small',
                                           device_map='cpu', torch_dtype=torch.float32)

context = torch.tensor(train.values, dtype=torch.float32)
quantiles, _ = pipe.predict_quantiles(context, prediction_length=H,
                                      quantile_levels=[0.1, 0.5, 0.9])
q = quantiles[0].numpy()          # (H, 3): p10, p50, p90
pred_chronos = pd.Series(q[:, 1], index=test.index)   # mediana

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train.index[-104:], train.values[-104:], color=BLUE, linewidth=1, alpha=0.6, label='train')
ax.plot(test.index, test.values, color='black', linewidth=2.5, label='REAL')
ax.plot(test.index, pred_chronos.values, color=PURPLE, linewidth=1.8, label='Chronos (zero-shot)')
ax.fill_between(test.index, q[:, 0], q[:, 2], color=PURPLE, alpha=0.15, label='intervalo p10–p90')
ax.axvline(train.index[-1], color='black', linestyle='--', linewidth=1)
ax.yaxis.set_major_formatter(unit_fmt)
ax.set_title('Chronos: pronóstico zero-shot (sin entrenar) con incertidumbre', fontsize=13, fontweight='bold')
ax.legend(loc='upper left', ncol=2)
plt.tight_layout()
plt.show()

print(f'Chronos zero-shot: WAPE {wape(test, pred_chronos):.3f} | MASE {mase(test, pred_chronos, train, m):.2f}')
print('Sin ajustar un solo parámetro ni construir un solo feature.')

---
## 4. Zero-shot a escala: cualquier SKU, sin entrenar

La gran ventaja operativa: **el mismo modelo pronostica cualquier serie** sin re-entrenar. Ideal para
*cold start* (SKUs nuevos sin historia para ajustar un modelo propio) o para miles de series.

In [ ]:
skus = df.groupby('sku_id')['revenue'].sum().nlargest(2).index.tolist()
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
for ax, s in zip(axes, skus):
    ser = df[df['sku_id'] == s].groupby(pd.Grouper(key='date', freq='W-MON'))['units_sold'].sum().iloc[1:-1]
    tr, te = ser.iloc[:-H], ser.iloc[-H:]
    qs, _ = pipe.predict_quantiles(torch.tensor(tr.values, dtype=torch.float32),
                                   prediction_length=H, quantile_levels=[0.1, 0.5, 0.9])
    qn = qs[0].numpy()
    ax.plot(te.index, te.values, color='black', linewidth=2, label='real')
    ax.plot(te.index, qn[:, 1], color=PURPLE, linewidth=1.6, label='Chronos')
    ax.fill_between(te.index, qn[:, 0], qn[:, 2], color=PURPLE, alpha=0.15)
    ax.set_title(s, fontsize=11, fontweight='bold'); ax.legend()
fig.suptitle('El mismo modelo, sin entrenar, sobre distintos SKUs', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print('Cero entrenamiento por serie: perfecto para arranque en frío y para escalar a miles de SKUs.')

---
## 5. Veredicto honesto vs. todo lo anterior

La comparación que cierra la Fase 3, contra el benchmark (V11), el SARIMAX (V12) y el XGBoost (V15).

In [ ]:
final = pd.DataFrame([
    {'modelo': 'Seasonal Naive (V11)', 'WAPE': wape(test, snaive), 'MASE': mase(test, snaive, train, m), 'entrena': 'no'},
    {'modelo': 'SARIMAX (V12)',        'WAPE': 0.049, 'MASE': 0.56, 'entrena': 'sí (+exógenas)'},
    {'modelo': 'XGBoost global (V15)', 'WAPE': 0.068, 'MASE': 0.79, 'entrena': 'sí (+features)'},
    {'modelo': 'Chronos (zero-shot)',  'WAPE': wape(test, pred_chronos), 'MASE': mase(test, pred_chronos, train, m), 'entrena': 'NO (zero-shot)'},
]).set_index('modelo')

fig, ax = plt.subplots(figsize=(11, 4.5))
colors = [GREEN if v < 1 else RED for v in final['MASE']]
ax.barh(final.index[::-1], final['MASE'][::-1], color=colors[::-1])
ax.axvline(1.0, color='black', linestyle='--', linewidth=1.5, label='listón (seasonal naive)')
ax.set_title('MASE por modelo (< 1 le gana al benchmark)', fontsize=13, fontweight='bold')
ax.legend()
for i, v in enumerate(final['MASE'][::-1]):
    ax.text(v, i, f' {v:.2f}', va='center', fontweight='bold')
plt.tight_layout()
plt.show()

print(final.assign(WAPE=lambda d: (d.WAPE * 100).round(1)).to_string())
print('\nHonestidad (el cierre de la fase):')
print('  • Chronos NO le gana al SARIMAX/XGBoost afinados, ni al seasonal naive, en ESTA serie:')
print('    tiene estacionalidad anual fuerte + eventos/promos MX que el modelo no conoce.')
print('  • PERO lo hizo ZERO-SHOT, en segundos, sin features ni ajuste. Como punto de partida es')
print('    asombroso — y estos modelos mejoran rápido. El futuro pinta híbrido: foundation + dominio.')

---
## 6. Cuándo usarlos (y notas de contexto)

| Foundation models brillan cuando… | No son la mejor opción cuando… |
|---|---|
| **Cold start**: SKUs nuevos sin historia para ajustar | Tienes historia rica y exógenas que explotar |
| Necesitas pronosticar **miles** de series ya | La estructura de dominio (promo, calendario) manda |
| Cero tiempo/experiencia para modelar | Buscas el último punto de precisión |
| Quieres un baseline probabilístico en segundos | — |

**TimeGPT** (Nixtla) es similar pero vía API de pago; por eso aquí usamos Chronos (abierto y local). Y un
apunte histórico: antes de los transformers, el deep learning de series usaba **LSTM** y **N-BEATS** — hoy
desplazados en zero-shot porque un LSTM se entrena por serie, mientras que un transformer pre-entrenado
generaliza a cualquiera. Es la misma historia que en NLP.

---
## 7. Conclusiones y puente a la Fase 4

### Las reglas que te llevas

1. **Zero-shot es real y asombroso:** un pronóstico probabilístico decente en segundos, sin entrenar ni
   construir features.
2. **Pero "sin esfuerzo" ≠ "mejor".** Aquí no superó al SARIMAX/XGBoost con exógenas de dominio.
3. **Su killer app es el cold start y la escala:** SKUs nuevos o miles de series, ya.
4. **El futuro es híbrido:** foundation model como base universal + tu conocimiento de dominio (exógenas,
   jerarquía, reglas de negocio) encima.
5. **Mídelo, como todo.** Un modelo de moda tampoco tiene derecho a ganar por defecto.

### Fin de la Fase 3

Recorrimos baselines, ARIMA/SARIMAX, Prophet, intermitentes, ML, jerárquico y foundation models — cada uno
medido honestamente contra el benchmark. Pero hemos usado WAPE y MASE casi a ciegas. Es hora de tomarnos
las **métricas** en serio.

---

### Próximo video
**Video 18 — Métricas que importan: MAE, MAPE, WAPE, BIAS y Forecast Value Added**
Más allá del error: sesgo persistente, FVA (¿tu modelo realmente mejora al baseline?) y cuánto cuesta
cada punto de MAPE en inventario inmovilizado.